# Module 1: Data Exploration for Schema Design

Goal: understand the raw Online Retail II data well enough to make
informed, deliberate decisions about the star schema (grain, key
handling, category derivation, region grouping) — before writing any
ETL code.

In [2]:
# Imports and load
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/online_retail_II.csv")
print(f"Shape: {df.shape}")
print(df.columns.tolist())
df.head()

Shape: (1067371, 8)
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
print(df.dtypes)
print(f"\nDate range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


## Geography: Country Distribution

In [4]:
print(f"{df['Country'].nunique()} unique countries\n")
print(df["Country"].value_counts().head(20))

43 unique countries

Country
United Kingdom     981330
EIRE                17866
Germany             17624
France              14330
Netherlands          5140
Spain                3811
Switzerland          3189
Belgium              3123
Portugal             2620
Australia            1913
Channel Islands      1664
Italy                1534
Norway               1455
Sweden               1364
Cyprus               1176
Finland              1049
Austria               938
Denmark               817
Unspecified           756
Greece                663
Name: count, dtype: int64


## Customer ID Completeness

In [5]:
missing_customer = df["Customer ID"].isna().sum()
print(f"Missing Customer ID: {missing_customer:,} of {len(df):,} ({missing_customer/len(df):.1%})")

Missing Customer ID: 243,007 of 1,067,371 (22.8%)


## Cancellations, Returns, and Quantity/Price Data Quality

In [6]:
df["Invoice"] = df["Invoice"].astype(str)
cancellations = df["Invoice"].str.startswith("C")

print(f"Cancellation rows (Invoice starts with 'C'): {cancellations.sum():,} ({cancellations.mean():.1%})")
print(f"Negative quantity rows: {(df['Quantity'] < 0).sum():,}")
print(f"Cancellation rows WITH negative quantity: {(cancellations & (df['Quantity'] < 0)).sum():,}")
print(f"Cancellation rows WITHOUT negative quantity (unexpected): {(cancellations & (df['Quantity'] >= 0)).sum():,}")
print(f"Negative-quantity rows that are NOT cancellations (unexpected): {((df['Quantity'] < 0) & ~cancellations).sum():,}")

Cancellation rows (Invoice starts with 'C'): 19,494 (1.8%)
Negative quantity rows: 22,950
Cancellation rows WITH negative quantity: 19,493
Cancellation rows WITHOUT negative quantity (unexpected): 1
Negative-quantity rows that are NOT cancellations (unexpected): 3,457


In [7]:
print(f"Zero or negative price rows: {(df['Price'] <= 0).sum():,}")
df[df["Price"] <= 0].head(10)

Zero or negative price rows: 6,207


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom


## Product Descriptions: Sample for Category Derivation

In [8]:
print(f"Unique StockCodes: {df['StockCode'].nunique():,}")
print(f"Missing descriptions: {df['Description'].isna().sum():,}\n")

print("Random sample of descriptions:")
for d in df["Description"].dropna().sample(20, random_state=1):
    print(f"  {d}")

Unique StockCodes: 5,305
Missing descriptions: 4,382

Random sample of descriptions:
  PAPER CHAIN KIT VINTAGE CHRISTMAS
  SET/6 PURPLE BUTTERFLY T-LIGHTS
  BLUE SAVANNAH PICNIC HAMPER FOR 2
  BLUE FELT EASTER EGG BASKET
  RED 3 PIECE MINI DOTS CUTLERY SET
  CREAM HEART CARD HOLDER
  FAMILY ALBUM WHITE PICTURE FRAME
  POSY CANDY BAG
  WRAP I LOVE LONDON 
  VICTORIAN GLASS HANGING T-LIGHT
  HEART FILIGREE DOVE LARGE
  FOLKART ZINC HEART CHRISTMAS DEC
  SET OF 72 RETRO SPOT PAPER  DOILIES
  CHOC TRUFFLE GOLD TRINKET POT 
  DOG BOWL , CHASING BALL DESIGN
  CREAM SWEETHEART LETTER RACK
  36 PENCILS TUBE RED RETROSPOT
  PACK OF 72 RETROSPOT CAKE CASES
  FELT EGG COSY WHITE RABBIT 
  DOG BOWL CHASING BALL DESIGN


In [9]:
# Look at the most common words across descriptions, to spot natural category groupings
from collections import Counter
import re

words = []
for desc in df["Description"].dropna():
    words.extend(re.findall(r"[A-Z]{3,}", str(desc)))

word_counts = Counter(words)
print("Most common description words (candidates for category keywords):")
for word, count in word_counts.most_common(30):
    print(f"  {word}: {count:,}")

Most common description words (candidates for category keywords):
  SET: 115,030
  RED: 92,688
  BAG: 92,682
  HEART: 79,153
  PINK: 65,036
  RETROSPOT: 58,799
  VINTAGE: 55,618
  DESIGN: 54,232
  WHITE: 51,164
  BOX: 50,939
  CAKE: 46,128
  METAL: 45,489
  CHRISTMAS: 44,557
  BLUE: 42,183
  HANGING: 37,100
  LIGHT: 36,738
  SIGN: 35,479
  JUMBO: 35,186
  HOLDER: 35,183
  PACK: 32,368
  PAPER: 30,852
  LUNCH: 30,441
  SMALL: 30,244
  GLASS: 27,728
  TEA: 26,780
  CARD: 25,781
  DECORATION: 25,035
  WOODEN: 23,831
  CASES: 23,343
  BOTTLE: 23,142


## Findings Summary (fill in after reviewing the output above)

- Country distribution: [X]% UK, rest split across [Y] other countries
- Missing Customer ID: [X]% — decision: [e.g., keep as "Unknown Customer" placeholder vs. exclude]
- Cancellations vs. negative quantity: [do they align cleanly, or is there a mismatch to document?]
- Zero/negative prices: [X] rows — decision: [exclude / flag / investigate specific cases]
- Candidate category keywords from description word frequency: [list the ones that look like real product categories, e.g. CHRISTMAS, BAG, MUG, CANDLE, etc.]